# Arquitectura de la Red Neuronal Siamesa

Este notebook describe la arquitectura de la red Siamesa usada para verificación facial
en el sistema de control de acceso dual.

## Contexto

El modelo **no clasifica personas por nombre ni por etiqueta fija**.
Aprende una **función de similitud** entre dos imágenes de rostros:
si las imágenes pertenecen a la misma persona, el score se acerca a `1`;
si son de personas distintas, se acerca a `0`.

Esto permite registrar nuevos usuarios **sin reentrenar**: basta con agregar
sus imágenes de referencia al *support set*.
Durante la inferencia, la cara capturada por la cámara se compara contra las referencias
del usuario identificado por RFID.

## Parámetros

In [ ]:
# --- Parámetros editables ---

# Si True, imprime el resumen de capas de cada modelo
RUN_MODEL_SUMMARY = True

# Si True, ejecuta una pasada forward con tensores aleatorios
RUN_DUMMY_FORWARD_PASS = True

# Tamaño del batch usado en la prueba forward
DUMMY_BATCH_SIZE = 2

## Importaciones y configuración

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import tensorflow as tf

# Detectar la raíz del proyecto independientemente del directorio de trabajo
_cwd = Path.cwd()
if (_cwd / 'src').exists():
    project_root = _cwd
elif (_cwd.parent / 'src').exists():
    project_root = _cwd.parent
else:
    raise RuntimeError(
        f'No se encontró el directorio src/. '
        f'Directorio actual: {_cwd}'
    )

sys.path.insert(0, str(project_root))

from src.config import (
    INPUT_SHAPE,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    IMAGE_CHANNELS,
    DEFAULT_SIMILARITY_THRESHOLD,
)
from src.models.siamese_network import (
    EMBEDDING_SIZE,
    build_embedding_network,
    build_siamese_model,
    compile_siamese_model,
)

print(f'TensorFlow {tf.__version__}')
print('Importaciones completadas.')

## Entradas del modelo

La red Siamesa recibe **dos imágenes** en cada inferencia:

| Entrada   | Forma           | Descripción                                         |
|-----------|-----------------|-----------------------------------------------------|
| `image_a` | `(112, 112, 3)` | Imagen RGB — captura de cámara o imagen de prueba   |
| `image_b` | `(112, 112, 3)` | Imagen RGB — referencia del *support set*           |

Ambas imágenes se normalizan al rango `[0, 1]`.
La red produce un **score de similitud** en `[0, 1]`.

In [ ]:
# Constantes del proyecto usadas por la red
print(f'INPUT_SHAPE                 : {INPUT_SHAPE}')
print(f'IMAGE_HEIGHT                : {IMAGE_HEIGHT}')
print(f'IMAGE_WIDTH                 : {IMAGE_WIDTH}')
print(f'IMAGE_CHANNELS              : {IMAGE_CHANNELS}')
print(f'EMBEDDING_SIZE              : {EMBEDDING_SIZE}')
print(f'DEFAULT_SIMILARITY_THRESHOLD: {DEFAULT_SIMILARITY_THRESHOLD}')

## Red de Embedding

La red de embedding convierte una imagen de rostro `(112, 112, 3)` en un **vector numérico** de `128` dimensiones.

Ese vector es la representación compacta de la cara:
- Dos imágenes de la **misma persona** producen vectores similares (distancia L1 pequeña).
- Dos imágenes de **personas distintas** producen vectores alejados (distancia L1 grande).

In [ ]:
# Construir la red de embedding
embedding_network = build_embedding_network()

if RUN_MODEL_SUMMARY:
    embedding_network.summary()

## Bloques principales de la red de embedding

La red aplica cuatro bloques convolucionales en cascada, reduciendo la resolución espacial
desde `112 × 112` hasta `7 × 7` antes de proyectar a un vector denso de `128` dimensiones.

In [ ]:
# Tabla resumida de los bloques de la red de embedding
blocks = [
    {'Bloque': '1',         'Capas': 'Conv2D(64)  + BatchNorm + MaxPool2D', 'Salida espacial': '56 × 56', 'Canales': 64},
    {'Bloque': '2',         'Capas': 'Conv2D(128) + BatchNorm + MaxPool2D', 'Salida espacial': '28 × 28', 'Canales': 128},
    {'Bloque': '3',         'Capas': 'Conv2D(256) + BatchNorm + MaxPool2D', 'Salida espacial': '14 × 14', 'Canales': 256},
    {'Bloque': '4',         'Capas': 'Conv2D(256) + BatchNorm + MaxPool2D', 'Salida espacial': '7 × 7',   'Canales': 256},
    {'Bloque': 'Flatten',   'Capas': 'Flatten',                             'Salida espacial': '12 544',  'Canales': '—'},
    {'Bloque': 'Dense 256', 'Capas': 'Dense(256, relu)',                    'Salida espacial': '256',     'Canales': '—'},
    {'Bloque': 'Dropout',   'Capas': 'Dropout(0.3)',                        'Salida espacial': '256',     'Canales': '—'},
    {'Bloque': f'Embedding {EMBEDDING_SIZE}', 'Capas': f'Dense({EMBEDDING_SIZE}) — sin activación', 'Salida espacial': str(EMBEDDING_SIZE), 'Canales': '—'},
]

arch_df = pd.DataFrame(blocks)
display(arch_df)

## Modelo Siamés completo — pesos compartidos

Ambas entradas (`image_a` e `image_b`) pasan por **la misma red de embedding**.
Los pesos son **compartidos**: no hay dos redes separadas, sino una sola instancia reutilizada.

Esto garantiza comparaciones consistentes:
si una imagen cambia ligeramente de pose, el embedding cambia de la misma manera en ambas ramas.

In [ ]:
# Construir el modelo Siamés completo (incluye la red de embedding con pesos compartidos)
siamese_model = build_siamese_model()

if RUN_MODEL_SUMMARY:
    siamese_model.summary()

## Módulo de comparación

Tras generar los dos embeddings, el modelo calcula la similitud en tres pasos:

```
embedding_a  =  f(image_a)                   # vector de 128 dimensiones
embedding_b  =  f(image_b)                   # vector de 128 dimensiones
distance     =  |embedding_a − embedding_b|  # distancia L1, elemento a elemento
similarity   =  Dense(1, sigmoid)(distance)  # score en [0, 1]
```

La capa `Lambda` aplica la distancia L1 elemento a elemento.
La capa `Dense(1, sigmoid)` aprende a convertir esa distancia en un score de similitud.

In [ ]:
# Compilar el modelo con el optimizador y métricas del proyecto
siamese_model = compile_siamese_model(siamese_model)

# Mostrar la configuración de compilación
optimizer_config = siamese_model.optimizer.get_config()
lr = optimizer_config.get('learning_rate', 'N/A')
print('Modelo compilado correctamente.')
print(f'  Optimizador        : {siamese_model.optimizer.__class__.__name__}')
print(f'  Learning rate      : {lr}')
print(f'  Función de pérdida : {siamese_model.loss.__class__.__name__}')
print('  Métricas           : binary_accuracy, precision, recall')

## Pasada forward con tensores aleatorios

Esta prueba **no entrena el modelo** ni actualiza ningún peso.
Solo verifica que el grafo computacional está bien definido y que las formas de los tensores son correctas.
Los scores obtenidos son aleatorios y **no tienen significado** antes del entrenamiento.

In [ ]:
if RUN_DUMMY_FORWARD_PASS:
    # Tensores aleatorios que simulan un batch de imágenes normalizadas en [0, 1]
    dummy_a = tf.random.uniform((DUMMY_BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))
    dummy_b = tf.random.uniform((DUMMY_BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))

    # Pasada forward sin actualizar pesos
    output = siamese_model([dummy_a, dummy_b], training=False)

    print('--- Formas de tensores ---')
    print(f'  dummy_a shape : {dummy_a.shape}')
    print(f'  dummy_b shape : {dummy_b.shape}')
    print(f'  output shape  : {output.shape}')
    print()
    print('--- Scores de similitud (aleatorios — sin entrenamiento) ---')
    for i, score in enumerate(output.numpy().flatten()):
        print(f'  muestra {i}: {score:.6f}')
else:
    print('Pasada forward deshabilitada (RUN_DUMMY_FORWARD_PASS = False).')

## Interpretación del score de similitud

| Score                             | Interpretación                                          |
|-----------------------------------|---------------------------------------------------------|
| Cercano a `1.0`                   | Imágenes muy similares — probablemente la misma persona |
| Cercano a `0.0`                   | Imágenes poco similares — personas distintas            |
| ≥ `DEFAULT_SIMILARITY_THRESHOLD`  | Decisión: **GRANTED**                                   |
| < `DEFAULT_SIMILARITY_THRESHOLD`  | Decisión: **DENIED**                                    |

> Los scores mostrados arriba son aleatorios y **no tienen significado** antes del entrenamiento.
> El umbral inicial es `0.5`; debe ajustarse usando datos de validación reales.

## Por qué esta arquitectura es adecuada para el proyecto

- **Trabaja con pares de imágenes**, no con etiquetas de persona fijas.
- **Soporta nuevos usuarios** sin reentrenar: basta con agregar referencias al *support set*.
- **Pesos compartidos** garantizan comparaciones consistentes entre vistas (`frontal`, `left`, `right`).
- **Multi-vista**: la cara capturada se compara contra todas las referencias del usuario y se toma el score máximo.
- **Compacto**: cuatro bloques convolucionales con embedding de `128` dimensiones — ejecutable en laptop.

## Lista de verificación

- [ ] El modelo se construye sin errores.
- [ ] Forma de entrada verificada: `(batch, 112, 112, 3)`.
- [ ] Forma de salida verificada: `(batch, 1)`.
- [ ] No se usan imágenes privadas ni datos personales.
- [ ] No se guarda ningún modelo ni checkpoint.
- [ ] Notebook listo para el script de entrenamiento (`src/training/`).
- [ ] **Outputs limpios antes del commit** (`Kernel → Restart & Clear Output`).